# 2 — NPC Neutral-Point Balancing

> **Goal.** Understand why the NPC's neutral point voltage **drifts
> open-loop**, derive the closed-loop carrier-based balancing
> controller that pins it back to $V_{dc}/2$, and demonstrate it on
> a forward-Euler switched simulation in pure Python.

This is the **defining control challenge** of the NPC topology —
without active balancing, the NP voltage walks away from its nominal
$V_{dc}/2$, eventually saturating one half of the DC bus and
destroying the 3-level operation.

**Prerequisites**

- [`01_npc_modeling.ipynb`](01_npc_modeling.ipynb) — switching states
  and PD-PWM.

**What you'll be able to do at the end**

1. State why the NP carries a real current and how its sign depends
   on the leg states + load currents.
2. Show that open-loop the NP voltage drifts (driven by 3rd-harmonic
   load-current content).
3. Implement a **carrier-based balancing controller**: a small DC
   offset added to all three references that biases the O-state dwell
   time and steers the NP back to its target.
4. Compare open-loop and closed-loop NP voltage trajectories on a
   forward-Euler simulation.


## Setup


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from npc_3phase_model import (
    NPC3PhaseParams,
    SWITCH_TABLE,
    switching_state_to_pole_voltage,
    pd_pwm_state,
    neutral_point_current,
)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

params = NPC3PhaseParams()
print(f"V_dc        = {params.V_dc} V   →  V_dc/2 = {params.V_dc_half} V (target NP)")
print(f"C_dc        = {params.C_dc*1e6:.0f} µF (each split cap)")
print(f"f_o         = {params.f_o} Hz,   f_sw = {params.f_sw/1e3:.0f} kHz")
print(f"R_load Y    = {params.R_load:.2f} Ω/phase")


## 1. Why does the NP drift?

The split DC bus is two capacitors $C_{dc}$ in series, with the
neutral point (NP) at their midpoint. In **steady state** the NP
sits at $V_{dc}/2$ — but only if the **net current** flowing into NP
averages to zero over one fundamental period.

A phase contributes to the NP current **only when it is in state O**.
In states P and N the load current routes around the NP entirely
(through the top half-bus or the bottom half-bus). In state O the
current passes through the clamping diodes to/from the NP:

$$
i_{NP}(t) = \sum_{\phi \in \{a,b,c\}} \mathbb{1}\left[\text{state}_\phi(t) = O\right]
\cdot i_\phi(t)
$$

where the indicator function is 1 only when phase $\phi$ is in state
O. The NP voltage evolves as:

$$
C_{dc} \, \frac{d V_{NP}}{dt} = i_{NP}(t)
\quad (\text{with appropriate sign convention})
$$

The key observation: **the time-average of $i_{NP}$ is generally not
zero**, even for a balanced load. The third-harmonic content of the
load current — which exists naturally in any nonlinear modulation
scheme — pumps net charge into one cap or the other.


In [ ]:
# Demonstrate symbolically: at m_a = 0.939 and a purely resistive
# load, evaluate the time-average of i_NP over one fundamental period.

def i_NP_signed(state_a, state_b, state_c, i_a, i_b, i_c):
    # Net current OUT of the neutral point. Sign convention: positive
    # flows from NP into the load. With our cap sign convention,
    # dV_NP/dt = -i_NP / C_dc.
    return neutral_point_current(state_a, state_b, state_c, i_a, i_b, i_c)


# Sample over 5 fundamental periods at high resolution.
fs = params.f_sw * 200
t = np.arange(0, 5.0 / params.f_o, 1 / fs)
omega = 2*np.pi*params.f_o

# References
v_ref_a = params.m_a * np.sin(omega * t)
v_ref_b = params.m_a * np.sin(omega * t - 2*np.pi/3)
v_ref_c = params.m_a * np.sin(omega * t - 4*np.pi/3)

# State-of-the-leg at each instant via PD-PWM
from npc_3phase_model import pd_pwm_state_vectorised
sa = pd_pwm_state_vectorised(v_ref_a, t, params.f_sw)
sb = pd_pwm_state_vectorised(v_ref_b, t, params.f_sw)
sc = pd_pwm_state_vectorised(v_ref_c, t, params.f_sw)

# Idealised load currents (sinusoidal at the operating point, in phase
# with v_ref_X for purely resistive load).
I_pk = params.I_o_pk
i_a = I_pk * np.sin(omega * t)
i_b = I_pk * np.sin(omega * t - 2*np.pi/3)
i_c = I_pk * np.sin(omega * t - 4*np.pi/3)

# Pulse-by-pulse i_NP — only the phase(s) in state O contribute.
i_NP = np.array([
    i_NP_signed(sa_i, sb_i, sc_i, ia_i, ib_i, ic_i)
    for sa_i, sb_i, sc_i, ia_i, ib_i, ic_i
    in zip(sa, sb, sc, i_a, i_b, i_c)
])

# Average i_NP over the last 3 fundamental periods (skip startup).
skip = int(2.0 / params.f_o / (t[1] - t[0]))
i_NP_avg = i_NP[skip:].mean()
print(f"  Average i_NP over the last 3 fundamental periods: "
      f"{i_NP_avg:.4f} A")
print(f"  Predicted dV_NP / dt = -i_NP / C_dc = "
      f"{-i_NP_avg / params.C_dc:.2f} V/s")
print(f"  → in one fundamental period the NP would drift by "
      f"{-i_NP_avg / params.C_dc / params.f_o * 1000:.2f} mV")
print()
print("  At fully balanced operation with purely sinusoidal references")
print("  the average is small but nonzero — and any operating asymmetry")
print("  amplifies it. The balancing controller below pins it to 0.")


## 2. Carrier-based balancing controller

The classic solution: add a **small DC offset** $v_{cm}$ to all three
references *common-mode* before the PD comparison:

$$
v_{ref,\phi}^{*}(t) = m_a \sin(\omega_o t - \theta_\phi) + v_{cm}
$$

The DC offset doesn't change the line-to-line output (it cancels in
$v_{ab} = v_{mid,a} - v_{mid,b}$) but it shifts each leg's O-state
dwell time *asymmetrically* with respect to the carrier positions,
which biases $\langle i_{NP} \rangle$.

A simple PI on the NP error gives the offset:

$$
v_{cm}(t) = K_p (V_{NP}^{\star} - V_{NP}) + K_i \int (V_{NP}^{\star} - V_{NP}) \, dt
$$

with the integrator winding clamped to avoid saturation.


In [ ]:
# Forward-Euler closed-loop simulation.
def simulate_npc_balancing(params, K_p=2e-3, K_i=1.0,
                              t_end=0.1, dt=2e-6,
                              v_np_init=None,
                              enable_balancer=True):
    # Run a switched NPC simulation with optional NP balancing.
    if v_np_init is None:
        v_np_init = params.V_dc_half + 5.0       # 5 V drift to start

    omega = 2*np.pi*params.f_o
    n_steps = int(t_end / dt) + 1

    v_np = v_np_init       # actual NP voltage (relative to vdc_neg)
    integ = 0.0
    R = params.R_load
    I_pk = params.I_o_pk

    # Record
    t_hist = np.zeros(n_steps)
    v_np_hist = np.zeros(n_steps)
    v_cm_hist = np.zeros(n_steps)
    i_NP_hist = np.zeros(n_steps)

    for k in range(n_steps):
        t = k * dt
        # PI on NP error
        err = params.V_dc_half - v_np
        if enable_balancer:
            integ += K_i * err * dt
            integ = float(np.clip(integ, -0.10, 0.10))   # ±10% clamp
            v_cm = K_p * err + integ
        else:
            v_cm = 0.0

        # References with common-mode offset.
        vra = params.m_a * np.sin(omega * t)              + v_cm
        vrb = params.m_a * np.sin(omega * t - 2*np.pi/3)  + v_cm
        vrc = params.m_a * np.sin(omega * t - 4*np.pi/3)  + v_cm

        # PD-PWM state.
        sa = pd_pwm_state(vra, t, params.f_sw)
        sb = pd_pwm_state(vrb, t, params.f_sw)
        sc = pd_pwm_state(vrc, t, params.f_sw)

        # Load currents (purely resistive in this simple sim).
        # Approximate v_pole - v_np as a proxy for the load voltage.
        va = switching_state_to_pole_voltage(sa, params.V_dc)
        vb = switching_state_to_pole_voltage(sb, params.V_dc)
        vc = switching_state_to_pole_voltage(sc, params.V_dc)
        ia = va / R
        ib = vb / R
        ic = vc / R

        # NP current.
        i_NP = neutral_point_current(sa, sb, sc, ia, ib, ic)

        # NP voltage integration. Two split caps in series → effective
        # cap to NP = C_dc/2 from either side. Net dV_NP/dt = -i_NP / C_dc_eff.
        # With C_dc_eff = C_dc (each cap, since they're symmetric):
        v_np -= (i_NP / params.C_dc) * dt

        t_hist[k] = t
        v_np_hist[k] = v_np
        v_cm_hist[k] = v_cm
        i_NP_hist[k] = i_NP

    return t_hist, v_np_hist, v_cm_hist, i_NP_hist


# Run two simulations: balancer OFF vs balancer ON.
t_o, vnp_o, vcm_o, i_np_o = simulate_npc_balancing(
    params, t_end=0.15, enable_balancer=False)
t_c, vnp_c, vcm_c, i_np_c = simulate_npc_balancing(
    params, t_end=0.15, enable_balancer=True, K_p=5e-3, K_i=2.0)

print(f"Open-loop  V_NP at t=150 ms: {vnp_o[-1]:.2f} V "
      f"(drift = {vnp_o[-1] - params.V_dc_half:+.2f} V from target {params.V_dc_half:.0f} V)")
print(f"Closed-loop V_NP at t=150 ms: {vnp_c[-1]:.2f} V "
      f"(drift = {vnp_c[-1] - params.V_dc_half:+.2f} V from target {params.V_dc_half:.0f} V)")


In [ ]:
fig, (ax_np, ax_cm) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax_np.plot(t_o * 1e3, vnp_o, color="C3", lw=1.2,
            label=f"Open-loop: V_NP drifts from {vnp_o[0]:.0f} V toward ...")
ax_np.plot(t_c * 1e3, vnp_c, color="C2", lw=1.2,
            label="Closed-loop with PI balancer (Kp=5e-3, Ki=2)")
ax_np.axhline(params.V_dc_half, color="k", ls="--", alpha=0.5,
                label=f"Target V_NP = V_dc/2 = {params.V_dc_half:.0f} V")
ax_np.set_ylabel("Neutral-point voltage [V]")
ax_np.set_title("NP voltage trajectory — open-loop drift vs closed-loop regulation")
ax_np.legend(loc="lower right")

ax_cm.plot(t_o * 1e3, vcm_o, color="C3", lw=1.2, label="open-loop ($v_{cm} = 0$)")
ax_cm.plot(t_c * 1e3, vcm_c, color="C2", lw=1.2, label="closed-loop $v_{cm}(t)$")
ax_cm.set_xlabel("Time [ms]")
ax_cm.set_ylabel("Common-mode offset $v_{cm}$ [-]")
ax_cm.legend(loc="upper right")

plt.tight_layout()
plt.show()


## 3. Summary

The NPC's neutral-point voltage is a **real state variable** that
the controller must actively regulate. Without intervention it
drifts away from $V_{dc}/2$, ultimately destroying the 3-level
operation.

The cure is mechanically simple — a PI on the NP error driving a
common-mode offset added to all three references — and costs nothing
in line-to-line output (the offset cancels in $v_{ab}$). The result
is a converter that maintains its 3-level signature indefinitely
even under unbalanced operation.

**Cross-validation note**: the
[`00_npc_pulsim_validation.ipynb`](00_npc_pulsim_validation.ipynb)
notebook uses a **stiff voltage-source DC bus** so the NP is rigidly
clamped — it can't drift in that model. To study the NP drift +
balancing in Pulsim you'd need to switch the bus to the
capacitor-only split, which currently runs into Pulsim's PWL cache
enumeration limits at this topology size. The pure-Python
forward-Euler simulation above is sufficient for studying the
balancing dynamics analytically.

**Suggested exercises**

1. Re-tune the balancer for **slower** dynamics (Kp = 1e-4, Ki = 0.2)
   and observe the longer settling time. What's the trade-off vs the
   load-cycle ripple amplitude?
2. Add a **load imbalance** (R_a ≠ R_b ≠ R_c) and observe how the
   open-loop NP drift accelerates. Can the same balancer handle it?
3. Replace the PI with a **deadbeat** controller (compute $v_{cm}$ to
   drive $i_{NP}$ to exactly zero in one switching period). Does it
   beat the PI on transient response?
